# Бенчмарк эмбеддингов — выбор модели для поиска по ТК РФ

**Роль 3 · Data Pipeline.** Сравниваем модели эмбеддингов по тому, насколько хорошо
они достают нужную статью ТК РФ по «житейскому» вопросу. Метрики: **Recall@1/3/5/10** и **MRR**.

- Корпус: актуальная редакция ТК РФ из ИПС (тот же `pipeline.fetch/parse/chunk`, что в проде).
- Вопросы: eval-набор Роли 2 (`tests/legal_cases/cases.json`).
- Всё считается **в памяти** (numpy-косинус) — Qdrant не нужен, ноутбук портируемый (Mac/ПК с GPU).

**Как запустить:** выбрать kernel = интерпретатор `~/.venvs/pocket-law-ai/bin/python`,
затем «Run All». Первый прогон скачает веса моделей (~4–5 ГБ) — прогресс виден в ячейках.

## 1. Окружение и устройство

In [ ]:
import sys, pathlib, time, json
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# repo root на sys.path, чтобы импортировать пакет pipeline (ноутбук лежит в pipeline/)
REPO = pathlib.Path.cwd()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

# Устройство: mps (Apple Silicon) / cuda (ПК с 4060 Ti) / cpu
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('repo :', REPO)
print('torch:', torch.__version__, '| device:', DEVICE)
print('ВНИМАНИЕ: время инференса на mps/cuda НЕ равно серверу Selectel (там только CPU).')
print('Качество (Recall/MRR) от устройства не зависит — цифры одинаковы везде.')

## 2. Корпус: ТК РФ (fetch → parse → chunk)

Тянем один раз, кэшируем в `pipeline/.state/corpus_tk_rf.json` (в git не попадает).
Повторный запуск ноутбука берёт из кэша — мгновенно.

In [ ]:
from pipeline.acts import ACTS
from pipeline.fetch import fetch_act
from pipeline.parse import parse_ips_html
from pipeline.chunk import chunk_article

CACHE = REPO / 'pipeline' / '.state' / 'corpus_tk_rf.json'

if CACHE.exists():
    corpus = json.loads(CACHE.read_text())
    print(f'из кэша: {len(corpus)} чанков')
else:
    info = ACTS['tk_rf']
    fetched = fetch_act(info)
    articles = parse_ips_html(fetched.html, act=info.act, min_articles=info.min_articles)
    corpus = []
    for a in articles:
        for c in chunk_article(a, fetched.revision_date):
            corpus.append({'article_no': a.article_no, 'text': c.text, 'status': a.status})
    CACHE.parent.mkdir(exist_ok=True)
    CACHE.write_text(json.dumps(corpus, ensure_ascii=False))
    print(f'скачано и распарсено: {len(corpus)} чанков из {len(articles)} статей, ред. {fetched.revision_date}')

passages = [c['text'] for c in corpus]
chunk_articles = [c['article_no'] for c in corpus]
print('уникальных статей в корпусе:', len(set(chunk_articles)))
print('пример чанка:', passages[0][:120], '...')

## 3. Eval-набор (вопросы Роли 2)

18 вопросов с целевой статьёй + 1 негативный (про погоду) — он не про право,
его держим отдельно: для эмбеддингов важно, чтобы у него был **низкий** максимум похожести
(сигнал для `insufficient_context` у Роли 2), в Recall он не входит.

In [ ]:
cases = json.loads((REPO / 'tests' / 'legal_cases' / 'cases.json').read_text())['cases']

eval_pos, eval_neg = [], []
for c in cases:
    arts = c.get('expected', {}).get('articles') or []
    (eval_pos if arts else eval_neg).append((c['question'], set(arts)))

print(f'позитивных: {len(eval_pos)} | негативных: {len(eval_neg)}')
for q, arts in eval_pos:
    print(f'  ст.{sorted(arts)!s:8} <- {q[:66]}')

## 4. Модели и их префиксы

Префиксы критичны для качества — у каждой семьи свои (сверено по карточкам моделей):
- **e5** (`intfloat/multilingual-e5-*`): `query:` / `passage:`
- **USER-bge-m3** (`deepvk/*`): без префикса (BGE-M3-стиль)
- (`ai-forever/ru-en-RoSBERTa`: `search_query:` / `search_document:` — закомментирована, включить одной строкой)

Добавить/убрать модель — правкой словаря `MODELS`.

In [ ]:
# name -> (hf_id, query_prefix, passage_prefix)
MODELS = {
    'e5-base (baseline)':  ('intfloat/multilingual-e5-base',  'query: ',  'passage: '),
    'e5-large':            ('intfloat/multilingual-e5-large', 'query: ',  'passage: '),
    'USER-bge-m3':         ('deepvk/USER-bge-m3',             '',         ''),
    # 'ru-en-RoSBERTa':    ('ai-forever/ru-en-RoSBERTa', 'search_query: ', 'search_document: '),
}
len(MODELS), list(MODELS)

## 5. Метрики

Один вопрос может «попасть» в несколько чанков одной статьи — сначала
ранжируем чанки по косинусу, потом схлопываем в **упорядоченный список уникальных статей**
и берём позицию первой ожидаемой статьи.

In [ ]:
def ranked_articles(sim_row):
    seen = []
    for idx in np.argsort(-sim_row):
        a = chunk_articles[idx]
        if a not in seen:
            seen.append(a)
    return seen

def first_hit_rank(sim_row, expected):
    for rank, a in enumerate(ranked_articles(sim_row), start=1):
        if a in expected:
            return rank
    return None  # не нашлось вообще

KS = (1, 3, 5, 10)

## 6. Прогон моделей

Тяжёлая ячейка: качает веса (первый раз) и считает эмбеддинги корпуса.
Для каждой модели меряем время кодирования корпуса на текущем устройстве.

In [ ]:
def evaluate(name, hf_id, q_pref, p_pref):
    model = SentenceTransformer(hf_id, device=DEVICE)
    t0 = time.perf_counter()
    pas = model.encode([p_pref + t for t in passages], batch_size=64,
                       normalize_embeddings=True, show_progress_bar=True)
    enc_sec = time.perf_counter() - t0

    qs = [q for q, _ in eval_pos]
    qemb = model.encode([q_pref + q for q in qs], normalize_embeddings=True)
    sims = qemb @ pas.T  # косинус (векторы нормированы)

    ranks = [first_hit_rank(sims[i], exp) for i, (_, exp) in enumerate(eval_pos)]
    row = {'model': name, 'dim': pas.shape[1]}
    for k in KS:
        row[f'R@{k}'] = float(np.mean([r is not None and r <= k for r in ranks]))
    row['MRR'] = float(np.mean([1.0 / r if r else 0.0 for r in ranks]))
    row['enc_sec'] = round(enc_sec, 1)
    row[f'enc/1k_{DEVICE}'] = round(enc_sec / len(passages) * 1000, 1)

    # негативный кейс: максимум похожести должен быть заметно ниже, чем у релевантных
    if eval_neg:
        nq = model.encode([q_pref + eval_neg[0][0]], normalize_embeddings=True)
        row['neg_max_sim'] = round(float((nq @ pas.T).max()), 3)
    del model
    return row, ranks

results, ranks_by_model = [], {}
for name, (hf_id, qp, pp) in MODELS.items():
    print(f'\n=== {name}  ({hf_id}) ===')
    row, ranks = evaluate(name, hf_id, qp, pp)
    results.append(row)
    ranks_by_model[name] = ranks
    print({k: row[k] for k in ('R@1','R@5','MRR','enc_sec')})

## 7. Итоговая таблица

In [ ]:
df = pd.DataFrame(results).set_index('model')
df = df.sort_values(['R@5', 'MRR'], ascending=False)
pd.set_option('display.width', 200)
df.style.format({c: '{:.2f}' for c in ['R@1','R@3','R@5','R@10','MRR']}) \
        .background_gradient(cmap='Greens', subset=['R@5','MRR'])

## 8. Разбор по вопросам

Позиция ожидаемой статьи у каждой модели (`—` = не попала даже в топ-10).
Особое внимание — трудная **ст. 81** (увольнение в отпуске): на ней спотыкался baseline.

In [ ]:
per_q = {}
for name, ranks in ranks_by_model.items():
    per_q[name] = ['—' if r is None else r for r in ranks]
drill = pd.DataFrame(per_q)
drill.insert(0, 'ст.', [sorted(exp)[0] for _, exp in eval_pos])
drill.insert(1, 'вопрос', [q[:55] for q, _ in eval_pos])
drill

## 9. Вывод и следующий шаг

Выбираем модель по цифрам (Recall@5 / MRR), с поправкой на CPU-стоимость на сервере
(колонка `enc/1k` — во сколько раз модель дороже base; на Selectel это только CPU).

**Когда модель выбрана:**
1. `EMBED_MODEL=<выбор>` в `.env` (Роль 4) и в `pipeline/config.py` (default).
2. Перезалить корпус: `python -m pipeline.run --act tk_rf --force` (создаст векторы новой размерности).
3. Роль 4: `search_law` кодирует запрос той же моделью с тем же префиксом.
4. Дописать результат в `STATUS.md` (секция Роли 3) и закоммитить.

Eval-набор и метрики отсюда потом заберёт Роль 2 в `tests/legal_cases` для автотеста качества.